In [1]:
from Complex_bug_impurity_solver import *
from utils import *
from Tree_topology import *
import scipy.sparse.linalg
from numpy import linalg
from matplotlib.pyplot import subplots, show

from pytreenet.ttns.ttns_ttno.application import (
    apply_ttno_to_ttns,
    ApplicationMethod,
)
import matplotlib.pyplot as plt
from TreeTopologies import random_impurity_binary_tree_ttn

In [2]:
from copy import deepcopy



nb = 2
D = 1.0

v, e = Discretisation_function(D, nb)

U = 2.0
U_prime = 3.0
J = 4.0
virtual_bond= 10
max_bond_dim = 10
angle_factor=-np.pi/50


# Fixed physical/JW ordering

physical_order = []

# Impurity modes
for orbital in range(1, 4):
    for spin in ("up", "down"):
        physical_order.append(
            f"Impurity_{orbital}_{spin}"
        )

# Bath modes
for orbital in range(1, 4):
    for bath_index in range(nb):
        for spin in ("up", "down"):
            physical_order.append(
                f"Bath_{orbital}_{spin}_{bath_index}"
            )

auxiliary_order = [
    "node1",
    "node2",
    "node3",
    "node4",
]

full_order = (
    physical_order
    + auxiliary_order
)

print("Physical/JW order:")
for i, site in enumerate(physical_order):
    print(i, site)


# Initial TTNS


tree_with_bath = T3NS(virtual_bond,bath=True,)

id_list = [
    "Bath_1_up_0",
    "Bath_1_down_0",
    "Bath_2_up_0",
    "Bath_2_down_0",
    "Bath_3_up_0",
    "Bath_3_down_0",
]

bath_list = []

for parent_id in id_list:
    bath_list.append(
        add_bath_chain(
            ttn=tree_with_bath,
            parent_id=parent_id,
            number_bath_sites=nb,
            bond_dim=5,
        )
    )

tree_with_bath.normalise()

# Complex-time TTNO


HK_hamil_with_bath = (
    construct_Hubbard_Kanamori_hamiltonian(
        3,
        U,
        U_prime,
        J,
        angle_factor=angle_factor,
    )
)

add_bath_interaction(
    HK_hamil_with_bath,
    nb,
    v,
    e,
)
HK_hamil_with_bath_physical= (
    construct_Hubbard_Kanamori_hamiltonian(
        3,
        U,
        U_prime,
        J,
        angle_factor=0.0,
    )
)

add_bath_interaction(
    HK_hamil_with_bath_physical,
    nb,
    v,
    e,
)
ttno_bath = (TreeTensorNetworkOperator.from_hamiltonian(hamiltonian=HK_hamil_with_bath,reference_tree=tree_with_bath,method=TTNOFinder.SGE))
ttno_bath_physical= (TreeTensorNetworkOperator.from_hamiltonian(hamiltonian=HK_hamil_with_bath_physical,reference_tree=tree_with_bath,method=TTNOFinder.SGE))


# ============================================================
# Physical sparse Hamiltonian
# ============================================================

HK_sparse_with_bath = (
    Hubbard_Kanamori_interaction_with_bath_sparse(
        3,
        nb,
        U,
        U_prime,
        J,
        v,
        e,
    )
)


print(
    "Sparse Hermiticity error:",
    scipy.sparse.linalg.norm(
        HK_sparse_with_bath
        - HK_sparse_with_bath.getH()
    ),
)


# ============================================================
# Complex-time Krylov snapshots
# ============================================================

time_step_size = 0.01
krylov_interval = 1.0
M = 10

Krylov_vectors = [tree_with_bath]
operators = {
    "energy": ttno_bath_physical,
}
current_state = deepcopy(
    tree_with_bath
)

Krylov_states= Complex_time_evolution(tree_with_bath,ttno_bath,time_step_size,krylov_interval,max_bond_dim,operators,M)



# Orthogonalise Krylov space


X, eig= Gram_schmidt_TTN(Krylov_states,tolerance= 1e-10,Keep_all=True)

print(
    "Number of snapshots:",
    len(Krylov_states),
)

print(
    "Orthogonalization shape:",
    X.shape,
)



# Physical Hamiltonian


(E_krylov,Q,coefficients,residuals,relative_residuals) = krylov_projection_TTN(ttno_bath_physical,X,Krylov_states,Keep_all=True)

sorting = np.argsort(np.real(E_krylov))

E_krylov = E_krylov[sorting]
Q = Q[:, sorting]

print("\nKrylov:")
print(E_krylov)


Physical/JW order:
0 Impurity_1_up
1 Impurity_1_down
2 Impurity_2_up
3 Impurity_2_down
4 Impurity_3_up
5 Impurity_3_down
6 Bath_1_up_0
7 Bath_1_down_0
8 Bath_1_up_1
9 Bath_1_down_1
10 Bath_2_up_0
11 Bath_2_down_0
12 Bath_2_up_1
13 Bath_2_down_1
14 Bath_3_up_0
15 Bath_3_down_0
16 Bath_3_up_1
17 Bath_3_down_1
Bath_1_up_1
Bath_1_down_1
Bath_2_up_1
Bath_2_down_1
Bath_3_up_1
Bath_3_down_1
Sparse Hermiticity error: 0.0
Iteration 1 out of 10


100%|██████████| 101/101 [01:16<00:00,  1.32it/s]


Iteration 2 out of 10


100%|██████████| 101/101 [01:21<00:00,  1.24it/s]


Iteration 3 out of 10


...

Iteration 9 out of 10


100%|██████████| 101/101 [01:32<00:00,  1.10it/s]


Iteration 10 out of 10


100%|██████████| 101/101 [01:31<00:00,  1.10it/s]


Number of snapshots: 11
Orthogonalization shape: (11, 11)
Ritz values and residuals:
 0: E = -3.6318833273, residual = 1.472e+00, relative = 4.054e-01
 1: E = -2.8841855279, residual = 1.904e+00, relative = 6.600e-01
 2: E = -1.9923344092, residual = 2.505e+00, relative = 1.257e+00
 3: E = -1.2544081431, residual = 2.396e+00, relative = 1.910e+00
 4: E = -0.6810911555, residual = 2.503e+00, relative = 2.503e+00
 5: E = -0.1591550474, residual = 2.738e+00, relative = 2.738e+00
 6: E =  0.3668301067, residual = 3.274e+00, relative = 3.274e+00
 7: E =  0.9845549813, residual = 3.919e+00, relative = 3.919e+00
 8: E =  1.8115183822, residual = 4.660e+00, relative = 2.573e+00
 9: E =  3.1535330185, residual = 5.262e+00, relative = 1.669e+00
10: E =  5.3720217095, residual = 5.758e+00, relative = 1.072e+00

Krylov:
[-3.63188333 -2.88418553 -1.99233441 -1.25440814 -0.68109116 -0.15915505
  0.36683011  0.98455498  1.81151838  3.15353302  5.37202171]


In [3]:
# Sparse ED reference


eigenvalues, eigenvectors = (scipy.sparse.linalg.eigsh(HK_sparse_with_bath,k=M,which="SA",))

eigenvalues = np.sort(
    np.real_if_close(eigenvalues)
)

print("\nED:")
print(eigenvalues)

print("\nKrylov:")
print(E_krylov)


ED:
[-5.1789599  -5.1789599  -5.1789599  -5.1789599  -4.92343933 -4.92343933
 -4.92343933 -4.92343933 -4.92343933 -4.92343933]

Krylov:
[-3.63188333 -2.88418553 -1.99233441 -1.25440814 -0.68109116 -0.15915505
  0.36683011  0.98455498  1.81151838  3.15353302  5.37202171]


In [ ]:
omega = np.linspace(-5 * U,5 * U,2000)

eta = 0.05 * U

G_krylov = np.array([Greens_function_lehman_TTN(E_krylov,coefficients,Krylov_states,eta,w) for w in omega])
A_krylov= spectral_function(G_krylov)
fig, ax = plt.subplots(1,1,figsize=(8, 6))
ax.set_xlabel('Frequency w/D', fontsize=10)
ax.set_ylabel("Spectral function A(w)",fontsize=10)
ax.plot(omega,A_krylov)
plt.xlim(-5,5)